In [1]:
import math
import random
from collections import deque
import pygame

pygame.mixer.pre_init(22050, -16, 1, 512)
pygame.init()

WIDTH, HEIGHT = 900, 600
FPS = 60
screen = pygame.display.set_mode((WIDTH, HEIGHT))
canvas = pygame.Surface((WIDTH, HEIGHT))
pygame.display.set_caption("Neon Pong")
clock = pygame.time.Clock()

title_font = pygame.font.SysFont("arial", 120, bold=True)
huge_font = pygame.font.SysFont("arial", 170, bold=True)
big_font = pygame.font.SysFont("arial", 54, bold=True)
font = pygame.font.SysFont("arial", 30)
small_font = pygame.font.SysFont("arial", 21)

WHITE = (255, 255, 255)
GOLD = (255, 215, 0)
GREY = (150, 155, 190)
GREEN = (90, 225, 130)
YELLOW = (255, 205, 70)
RED = (255, 95, 95)
P1_COLOR = (0, 220, 255)
P2_COLOR = (255, 80, 160)
BG_TOP = (12, 14, 40)
BG_BOTTOM = (42, 16, 66)
BG_MID = (27, 15, 53)

PADDLE_W, PADDLE_H = 16, 100
PADDLE_SPEED = 9
LEFT_X = 40
RIGHT_X = WIDTH - 40 - PADDLE_W
DIFFICULTIES = ["Easy", "Medium", "Hard"]
MIN_TARGET, MAX_TARGET = 1, 21


def mix(c1, c2, t):
    t = max(0, min(1, t))
    return tuple(int(c1[i] + (c2[i] - c1[i]) * t) for i in range(3))


def draw_text(text, f, color, center, scale=1.0):
    if not text:
        return
    img = f.render(text, True, color)
    if scale != 1.0:
        img = pygame.transform.rotozoom(img, 0, scale)
    canvas.blit(img, img.get_rect(center=center))


def make_background():
    surf = pygame.Surface((WIDTH, HEIGHT))
    for y in range(HEIGHT):
        color = mix(BG_TOP, BG_BOTTOM, y / HEIGHT)
        pygame.draw.line(surf, color, (0, y), (WIDTH, y))
    return surf

def make_circle_glow(radius, color, layers=10):   
    surf = pygame.Surface((radius * 2, radius * 2), pygame.SRCALPHA)   
    for i in range(layers, 0, -1):          
        r = int(radius * i / layers)       
        alpha = int(6 + 80 * (1 - i / layers) ** 1.5)   
        pygame.draw.circle(surf, (*color, alpha), (radius, radius), r)
    return surf

def make_rect_glow(w, h, color, layers=8, step=3): 
    pad = layers * step                     
    surf = pygame.Surface((w + 2 * pad, h + 2 * pad), pygame.SRCALPHA)  
    for i in range(layers, 0, -1):          
        grow = i * step                     
        rect = pygame.Rect(pad - grow, pad - grow, w + 2 * grow, h + 2 * grow)   
        alpha = int(70 * (1 - i / (layers + 1)))   
        pygame.draw.rect(surf, (*color, alpha), rect, border_radius=8 + grow)   
    return surf, pad  

background = make_background()             
ball_glow = make_circle_glow(34, (130, 170, 255))   
dim = pygame.Surface((WIDTH, HEIGHT), pygame.SRCALPHA)   
dim.fill((8, 8, 24, 175))                  

def make_sound(notes, volume=0.2):
    sample_rate = 22050                   
    data = bytearray()                    
    for freq, ms in notes:                  
        n_samples = int(sample_rate * ms / 1000)   
        for i in range(n_samples):        
            attack = min(1, i / 200)      
            decay = math.exp(-4 * i / n_samples)  
            wave = math.sin(2 * math.pi * freq * i / sample_rate) 
            value = int(volume * 32767 * attack * decay * wave)    
            data += value.to_bytes(2, "little", signed=True)      
    return pygame.mixer.Sound(buffer=bytes(data))   

try:
    snd_move = make_sound([(600, 45)], 0.10)                                  
    snd_select = make_sound([(700, 60), (930, 110)], 0.14)                    
    snd_hit = make_sound([(480, 90)], 0.20)                                   
    snd_wall = make_sound([(330, 60)], 0.12)                                  
    snd_score = make_sound([(523, 90), (392, 220)], 0.18)                     
    snd_win = make_sound([(523, 120), (659, 120), (784, 120), (1047, 380)], 0.2)   
    snd_count = make_sound([(500, 110)], 0.14)                               
    snd_go = make_sound([(800, 220)], 0.16)                                   
    sound_on = True                        
except Exception:
    sound_on = False   
    snd_move = snd_select = snd_hit = snd_wall = None     
    snd_score = snd_win = snd_count = snd_go = None


def play(sound):
    if sound_on and sound is not None:      
        sound.play()                      


class Paddle:
    def __init__(self, x, color):          
        self.rect = pygame.Rect(x, HEIGHT // 2 - PADDLE_H // 2, PADDLE_W, PADDLE_H)   
        self.y = float(self.rect.y)         
        self.color = color                  
        self.flash = 0                       
        self.glow, self.pad = make_rect_glow(PADDLE_W, PADDLE_H, color) 

    def set_y(self, y):
        self.y = max(10, min(HEIGHT - 10 - PADDLE_H, y))   
        self.rect.y = int(round(self.y))    

    def move(self, dy):
        self.set_y(self.y + dy)           

    def set_center(self, cy):
        self.set_y(cy - PADDLE_H / 2)      

    def draw(self, surf):                 
        surf.blit(self.glow, (self.rect.x - self.pad, self.rect.y - self.pad))  
        color = mix(self.color, WHITE, self.flash / 10)   
        pygame.draw.rect(surf, color, self.rect, border_radius=8)   
        if self.flash > 0:                 
            self.flash -= 1               


class Ball:                                
    RADIUS = 10                             
    START_SPEED = 7                        
    MAX_SPEED = 15                         

    def __init__(self):                    
        self.trail = deque(maxlen=14)       
        self.hit_paddle = None             
        self.reset(1)                       

    def reset(self, direction):             
        self.x = WIDTH / 2                  
        self.y = HEIGHT / 2                
        self.speed = self.START_SPEED       
        angle = random.uniform(-0.5, 0.5)   
        self.vx = direction * self.speed * math.cos(angle)   
        self.vy = self.speed * math.sin(angle)               
        self.trail.clear()                  

    def bounce(self, paddle, direction):    
        offset = (self.y - paddle.rect.centery) / (paddle.rect.height / 2)  
        offset = max(-1, min(1, offset))   
        angle = offset * math.radians(55)   
        self.speed = min(self.speed * 1.07, self.MAX_SPEED)   
        self.vx = direction * self.speed * math.cos(angle)    
        self.vy = self.speed * math.sin(angle)                
        paddle.flash = 10                  
        self.hit_paddle = paddle            

    def update(self, left, right):         
        self.trail.append((self.x, self.y)) 
        self.x += self.vx                
        self.y += self.vy                 
        r = self.RADIUS                   
        event = None                      

        if self.y < r:                    
            self.y = r                    
            self.vy = abs(self.vy)          
            event = "wall"                
        elif self.y > HEIGHT - r:          
            self.y = HEIGHT - r            
            self.vy = -abs(self.vy)        
            event = "wall"

        ball_rect = pygame.Rect(self.x - r, self.y - r, 2 * r, 2 * r)  
        if self.vx < 0 and self.x > left.rect.centerx and ball_rect.colliderect(left.rect):
            self.x = left.rect.right + r   
            self.bounce(left, 1)          
            event = "paddle"
        elif self.vx > 0 and self.x < right.rect.centerx and ball_rect.colliderect(right.rect):
            self.x = right.rect.left - r   
            self.bounce(right, -1)         
            event = "paddle"

        if self.x < -2 * r:               
            event = "right_scores"        
        elif self.x > WIDTH + 2 * r:        
            event = "left_scores"           
        return event                      

    def update_free(self):
        self.trail.append((self.x, self.y)) 
        self.x += self.vx                 
        self.y += self.vy                
        r = self.RADIUS                 
        if self.x < r:                    
            self.x = r
            self.vx = abs(self.vx)        
        elif self.x > WIDTH - r:         
            self.x = WIDTH - r
            self.vx = -abs(self.vx)        
        if self.y < r:                    
            self.y = r
            self.vy = abs(self.vy)         
        elif self.y > HEIGHT - r:         
            self.y = HEIGHT - r
            self.vy = -abs(self.vy)       


def draw_ball(b):                           
    heat = (b.speed - Ball.START_SPEED) / (Ball.MAX_SPEED - Ball.START_SPEED)   
    color = mix(WHITE, (255, 190, 70), heat)           
    for i, (tx, ty) in enumerate(b.trail):  
        t = (i + 1) / (len(b.trail) + 1)    
        pygame.draw.circle(canvas, mix(BG_MID, color, t * 0.7), (int(tx), int(ty)), max(2, int(Ball.RADIUS * t)))
    canvas.blit(ball_glow, (int(b.x) - 34, int(b.y) - 34))   
    pygame.draw.circle(canvas, color, (int(b.x), int(b.y)), Ball.RADIUS)   

def predict_y(ball, x_target):              
    if ball.vx <= 0:                     
        return HEIGHT / 2                  
    t = (x_target - ball.x) / ball.vx       
    if t < 0:                             
        return ball.y                    
    top = Ball.RADIUS                     
    span = HEIGHT - 2 * Ball.RADIUS       
    y = (ball.y + ball.vy * t - top) % (2 * span)  
    if y > span:                            
        y = 2 * span - y                    
    return top + y


class Robot:                               
    SETTINGS = {"Easy": (4.5, 60, False),
                "Medium": (6.2, 40, True),
                "Hard": (8.2, 28, True)}    

    def __init__(self, level):           
        self.speed, self.max_error, self.predicts = self.SETTINGS[level]  
        self.new_error()                   

    def new_error(self):                    
        self.error = random.uniform(-self.max_error, self.max_error) 

    def update(self, paddle, ball, active):
        if active and ball.vx > 0:        
            if self.predicts:              
                target = predict_y(ball, paddle.rect.left - Ball.RADIUS) + self.error  
            else:                          
                target = ball.y + self.error   
        else:
            target = HEIGHT / 2                        
        diff = target - paddle.rect.centery 
        if abs(diff) > 6:                 
            paddle.move(max(-self.speed, min(self.speed, diff)))   


particles = []                             
confetti = []                               


def spark(x, y, color, count=14, speed=5):  
    for _ in range(count):                 
        angle = random.uniform(0, 2 * math.pi)  
        v = random.uniform(1, speed)       
        particles.append([x, y, math.cos(angle) * v, math.sin(angle) * v, 30, color])

def update_particles():                     
    for p in particles[:]:                 
        p[0] += p[2]                      
        p[1] += p[3]                        
        p[2] *= 0.96                       
        p[3] *= 0.96
        p[4] -= 1                          
        if p[4] <= 0:                   
            particles.remove(p)         
    for c in confetti[:]:                
        c[0] += c[2] + math.sin(c[1] * 0.05)  
        c[1] += c[3]                       
        if c[1] > HEIGHT + 10:            
            confetti.remove(c)           

def draw_particles():                      
    for p in particles:                 
        pygame.draw.circle(canvas, mix(BG_MID, p[5], p[4] / 30), (int(p[0]), int(p[1])), max(1, int(4 * p[4] / 30)))
    for c in confetti:                   
        pygame.draw.rect(canvas, c[4], (int(c[0]), int(c[1]), 7, 11))

left = Paddle(LEFT_X, P1_COLOR)          
right = Paddle(RIGHT_X, P2_COLOR)        
ball = Ball()                           
menu_ball = Ball()                  
menu_ball.vx, menu_ball.vy, menu_ball.speed = 5.0, 3.4, 9   
robot = None                              

state = "menu"         
                       
previous_state = "playing"  
mode = "robot"          
menu_index = 0        
difficulty_index = 1    
target_score = 5       
over_index = 0         
left_score = 0        
right_score = 0        
winner = None           
count_timer = 0         
count_numbers = True   
last_count = 99        
shake = 0              
flash = {"left": 0, "right": 0}   
frame = 0            
running = True         


def player_names():                         
    if mode == "robot":
        return "YOU", "ROBOT (" + DIFFICULTIES[difficulty_index] + ")"   
    return "PLAYER 1", "PLAYER 2"


def player_hints():                       
    if mode == "robot":
        return "W / S  or  UP / DOWN", ""  
    return "W / S", "UP / DOWN"


def winner_text(side):                      
    if mode == "robot":
        return "YOU WIN!" if side == "left" else "ROBOT WINS!"
    return "PLAYER 1 WINS!" if side == "left" else "PLAYER 2 WINS!"


def start_match():                          
    global state, robot, left_score, right_score, count_timer, count_numbers, last_count, winner
    left.set_center(HEIGHT / 2)            
    right.set_center(HEIGHT / 2)
    ball.reset(random.choice([-1, 1]))     
    left_score = 0                         
    right_score = 0
    winner = None                          
    robot = Robot(DIFFICULTIES[difficulty_index]) if mode == "robot" else None  
    particles.clear()                      
    confetti.clear()
    state = "countdown"                    
    count_timer = FPS * 3                   
    count_numbers = True                   
    last_count = 99                       


def point_scored(side):                   
    global left_score, right_score, state, winner, shake, count_timer, count_numbers, last_count, over_index
    if side == "left":                     
        left_score += 1                    
        spark(WIDTH - 5, min(max(ball.y, 20), HEIGHT - 20), P1_COLOR, 30, 8)   
    else:                                  
        right_score += 1
        spark(5, min(max(ball.y, 20), HEIGHT - 20), P2_COLOR, 30, 8)           
    flash[side] = 25                      
    shake = 14                             
    play(snd_score)                       

    if left_score >= target_score or right_score >= target_score:   
        winner = side                       
        state = "gameover"                 
        over_index = 0                      
        play(snd_win)                       
        ball.reset(1)                      
    else:                                  
        ball.reset(1 if side == "left" else -1)          
        state = "countdown"               
        count_timer = 50                   
        count_numbers = False            
        last_count = 99

def handle_key(key):                       
    global state, previous_state, mode, menu_index, difficulty_index, target_score, over_index, running
    up = key in (pygame.K_UP, pygame.K_w)          
    down = key in (pygame.K_DOWN, pygame.K_s)     
    minus = key in (pygame.K_LEFT, pygame.K_a)    
    plus = key in (pygame.K_RIGHT, pygame.K_d)     
    confirm = key in (pygame.K_RETURN, pygame.K_KP_ENTER, pygame.K_SPACE) 
    back = key == pygame.K_ESCAPE                  

    if state == "menu":                    
        if up or down:                     
            menu_index = (menu_index + (1 if down else -1)) % 3  
            play(snd_move)
        elif confirm:                      
            play(snd_select)
            if menu_index == 0:            
                mode = "robot"
                state = "difficulty"      
            elif menu_index == 1:         
                mode = "friend"
                state = "score"             
            else:                         
                running = False             
        elif back:                         
            running = False

    elif state == "difficulty":           
        if up or down:
            difficulty_index = (difficulty_index + (1 if down else -1)) % 3   
            play(snd_move)
        elif confirm:
            play(snd_select)
            state = "score"                
        elif back:
            state = "menu"               

    elif state == "score":                 
        if minus or plus or up or down:
            step = 1 if plus else -1 if minus else 5 if up else -5 
            target_score = max(MIN_TARGET, min(MAX_TARGET, target_score + step))  
            play(snd_move)
        elif confirm:
            play(snd_select)
            start_match()                   
        elif back:
            state = "difficulty" if mode == "robot" else "menu"   

    elif state in ("countdown", "playing"): 
        if back:                            
            previous_state = state         
            state = "paused"

    elif state == "paused":               
        if confirm or back:                
            state = previous_state
        elif key == pygame.K_m:            
            state = "menu"

    elif state == "gameover":             
        if up or down:
            over_index = 1 - over_index    
            play(snd_move)
        elif confirm:
            play(snd_select)
            if over_index == 0:            
                start_match()
            else:                           
                state = "menu"
        elif back:
            state = "menu"




pygame-ce 2.5.8 (SDL 2.32.10, Python 3.14.8)
